# Testing — load NEW vs OLD processed data (transunion + experian) and targets

## What the changes were

- **Denominator fix (`missing_data_chars`)**: the `percent_<rate>_<window>_months`
  features divided by months that were never observed — bureau missing-data
  codes (`*` equifax, `-` experian, `X` transunion) counted as observed
  paid-as-agreed months, biasing the rates toward 0. The fixed aggregator
  excludes those positions and anchors the denominator on the observed string
  length.
- **Experian placeholder change**: the asset's `placeholder: "-"` deleted
  dashes from the payment pattern, but the dash is a real month ("No update
  received", CIS Guide Appendix T) — stripping it shifted every older month
  one position more recent, misaligning windows and `months_since_*`
  features. The placeholder is removed on the `payment_processor_change`
  branch.

## Why experian needs its own NEW normalized + processed

The experiment's `processed_new` was built BEFORE the placeholder change, so
for experian it does NOT reflect the full new behavior — the dashes were
still being stripped. `NewProcessing_Experian_{Train,Test}.ipynb` re-ran
phase 1 (normalized) and phase 2 (processed) for the same 400k sampled
applicants with the branch code, writing to
`new_normalized_and_processed/experian_<role>/`. Both stages were needed:
normalized because the pattern string itself changes (dashes kept), and
processed because the per-applicant `trade_*` features are aggregated from
that normalized data.

For **transunion and equifax** the branch pipeline is behavior-identical to
the experiment's NEW run (TU's `/` placeholder never occurs in the data;
equifax is unchanged), so their `processed_new` IS the with-changes data —
no re-run needed. OLD = `processed_old` (shipping behavior) for everyone.

In [12]:
import pandas as pd

DATA = '/home/jag/payment-processor-research/payment_processing_research_data'

def load_processed(path):
    df = pd.read_parquet(path)   # reads every part-*.parquet in the dir
    print(f'{path.split("research_data/")[-1]:60} {df.shape}')
    return df

In [13]:
# transunion: NEW and OLD, train + test (processed_new is already
# with-changes for TU -- the branch is behavior-identical there)
transunion_train_new = load_processed(f'{DATA}/samples/transunion_train/processed_new')
transunion_train_old = load_processed(f'{DATA}/samples/transunion_train/processed_old')
transunion_test_new  = load_processed(f'{DATA}/samples/transunion_test/processed_new')
transunion_test_old  = load_processed(f'{DATA}/samples/transunion_test/processed_old')

samples/transunion_train/processed_new                       (395913, 8848)
samples/transunion_train/processed_old                       (395913, 8848)
samples/transunion_test/processed_new                        (395679, 8848)
samples/transunion_test/processed_old                        (395679, 8848)


In [14]:
# experian: NEW comes from new_normalized_and_processed (built with the
# placeholder change); OLD from processed_old as usual
experian_train_new = load_processed(f'{DATA}/new_normalized_and_processed/experian_train/processed')
experian_train_old = load_processed(f'{DATA}/samples/experian_train/processed_old')
experian_test_new  = load_processed(f'{DATA}/new_normalized_and_processed/experian_test/processed')
experian_test_old  = load_processed(f'{DATA}/samples/experian_test/processed_old')

new_normalized_and_processed/experian_train/processed        (394241, 8848)
samples/experian_train/processed_old                         (394241, 8848)
new_normalized_and_processed/experian_test/processed         (394447, 8848)
samples/experian_test/processed_old                          (394447, 8848)


In [15]:
# targets: all six (bureau, role) sample target files, tagged and stacked
target_paths = {
    ('equifax', 'train'):    f'{DATA}/samples/equifax_train/target.parquet',
    ('equifax', 'test'):     f'{DATA}/samples/equifax_test/target.parquet',
    ('experian', 'train'):   f'{DATA}/samples/experian_train/target.parquet',
    ('experian', 'test'):    f'{DATA}/samples/experian_test/target.parquet',
    ('transunion', 'train'): f'{DATA}/samples/transunion_train/target.parquet',
    ('transunion', 'test'):  f'{DATA}/samples/transunion_test/target.parquet',
}

targets = pd.concat(
    [pd.read_parquet(p).assign(bureau=b, role=r) for (b, r), p in target_paths.items()],
    ignore_index=True)
print('targets:', targets.shape)
print(targets.groupby(['bureau', 'role']).size())

targets: (2400000, 151)
bureau      role 
equifax     test     400000
            train    400000
experian    test     400000
            train    400000
transunion  test     400000
            train    400000
dtype: int64


In [16]:
# feature lists (same schema on every processed table)
percent_features = [f for f in transunion_train_new.columns
                    if 'percent_of_DQ' in f and 'in_last' in f.lower()]
number_features  = [f for f in transunion_train_new.columns
                    if 'in_last' in f.lower() and f not in percent_features]
print(f'{len(percent_features)} percent features, {len(number_features)} number/other in_last features')

1143 percent features, 1637 number/other in_last features


## Distribution drift — PSI (zaml standard)

PSI with OLD as the benchmark and NEW as the comparison, per feature:

- **psi_overall** — on all common applicants. Expect ~0 everywhere: only ~6%
  of rows change, so the population histogram barely moves.
- **psi_changed_rows** — restricted to the applicants whose value actually
  differs. This is where any real distributional movement shows.

Rule of thumb: < 0.1 no shift, 0.1–0.25 moderate, > 0.25 significant.

In [23]:
from zaml.analyze.data_analysis.distribution_drift import PSI 

In [26]:


def psi_table(new, old, features, label, group=10, min_changed=100):
    """PSI per feature, OLD as benchmark vs NEW as comparison."""
    idx   = new.index.intersection(old.index)
    feats = [c for c in features if c in new.columns and c in old.columns]
    n, o  = new.loc[idx, feats], old.loc[idx, feats]

    # overall PSI: fit benchmark on OLD, score NEW
    psi_obj = PSI(group=group)
    psi_obj.fit(o, columns=feats)
    print(f'fit')
    psi_all = psi_obj.transform(n)

    # PSI restricted to the changed applicants, per feature
    psi_chg, n_chg = {}, {}
    for c in feats:
        af, bf = n[c].astype('float64'), o[c].astype('float64')
        neq = ~np.isclose(af, bf, equal_nan=True)
        n_chg[c] = int(neq.sum())
        if n_chg[c] >= min_changed:          # need enough rows to bin sanely
            p = PSI(group=group)
            p.fit(o.loc[neq, [c]])
            psi_chg[c] = p.transform(n.loc[neq, [c]]).get(c, np.nan)

    out = pd.DataFrame({'psi_overall':      psi_all,
                        'psi_changed_rows': pd.Series(psi_chg),
                        'n_changed':        pd.Series(n_chg)})
    out['pct_changed'] = 100 * out['n_changed'] / len(idx)
    out = out.sort_values('psi_changed_rows', ascending=False)

    print(f'=== {label} (n={len(idx):,}) ===')
    print(out.head(20).round(4).to_string())
    print()
    return out

In [27]:
psi_tu_percent  = psi_table(transunion_train_new, transunion_train_old, percent_features, 'transunion train')

Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.


fit


Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The met

=== transunion train (n=395,913) ===
                                                                                                        psi_overall  psi_changed_rows  n_changed  pct_changed
trade_max_percent_of_DQ30_or_greater_in_last_6_months__derog_open_cc                                            0.0           24.5493        104       0.0263
trade_max_percent_of_DQ30_or_greater_in_last_6_months__non_derog_open_cc                                        0.0           24.3774        309       0.0780
trade_min_percent_of_DQ30_or_greater_in_last_6_months__derog_open_cc                                            0.0           24.3043        120       0.0303
trade_max_percent_of_DQ30_or_greater_in_last_6_months__derog_open_revolving                                     0.0           23.8988        147       0.0371
trade_min_percent_of_DQ30_or_greater_in_last_6_months__derog_open_revolving                                     0.0           23.5860        158       0.0399
trade_min_perce

In [33]:
psi_tu_percent.sort_values(by = ['n_changed', 'psi_overall'], ascending = False).head(50)

,psi_overall,psi_changed_rows,n_changed,pct_changed
trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts,2.683601e-03,0.204753,25863,6.532496
trade_mean_percent_of_DQ30_in_last_24_months__active_open_accounts,2.667953e-03,0.205161,25584,6.462026
trade_mean_percent_of_DQ30_in_last_24_months__non_derog_open_accounts,1.687402e-03,0.202535,23193,5.858105
trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts,1.354078e-02,9.244651,23013,5.812641
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts,1.371763e-02,8.842768,22842,5.769449
trade_mean_percent_of_DQ30_in_last_24_months__individual_open_accounts,1.530585e-03,0.279719,21556,5.444631
trade_max_percent_of_DQ30_in_last_24_months__non_derog_open_accounts,1.247631e-02,9.123610,20786,5.250143
trade_mean_percent_of_DQ30_in_last_24_months__with_recent_payment_open_accounts,1.799135e-03,0.247362,20749,5.240798
trade_max_percent_of_DQ30_in_last_24_months__individual_open_accounts,0.000000e+00,10.331031,19627,4.957402
trade_max_percent_of_DQ30_in_last_24_months__with_recent_payment_open_accounts,0.000000e+00,9.493168,18543,4.683605


In [34]:
psi_exp_percent = psi_table(experian_train_new, experian_train_old, percent_features, 'experian train')

Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.


fit


Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The met

=== experian train (n=394,241) ===
                                                                                                 psi_overall  psi_changed_rows  n_changed  pct_changed
trade_min_percent_of_DQ30_or_greater_in_last_6_months__derog_open_cc                                     0.0           45.9991        108       0.0274
trade_max_percent_of_DQ30_or_greater_in_last_6_months__derog_open_cc                                     0.0           45.9957        100       0.0254
trade_min_percent_of_DQ60_in_last_12_months__non_derog_open_cc                                           0.0           23.9183        113       0.0287
trade_min_percent_of_DQ60_in_last_24_months__non_derog_open_cc                                           0.0           21.2280        513       0.1301
trade_mean_percent_of_DQ30_or_greater_in_last_6_months__derog_open_cc                                    0.0           20.2698        117       0.0297
trade_min_percent_of_DQ60_in_last_24_months__derog_open_cc 

In [38]:
psi_exp_percent.sort_values(by = ['n_changed', 'psi_overall'], ascending = False).head(50)

,psi_overall,psi_changed_rows,n_changed,pct_changed
trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts,2.463591e-03,0.174536,26129,6.627672
trade_mean_percent_of_DQ30_in_last_24_months__active_open_accounts,2.466081e-03,0.178532,25936,6.578717
trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts,1.331175e-02,2.166650,22970,5.826385
trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts,1.345264e-02,1.730454,22842,5.793918
trade_mean_percent_of_DQ30_in_last_24_months__non_derog_open_accounts,1.724808e-03,0.203694,22094,5.604186
trade_mean_percent_of_DQ30_in_last_24_months__individual_open_accounts,1.670655e-03,0.247102,21492,5.451488
trade_mean_percent_of_DQ30_in_last_24_months__with_recent_payment_open_accounts,1.583173e-03,0.194591,20654,5.238927
trade_max_percent_of_DQ30_in_last_24_months__non_derog_open_accounts,1.469823e-06,1.986187,19961,5.063147
trade_max_percent_of_DQ30_in_last_24_months__individual_open_accounts,1.018305e-06,2.298246,19348,4.907658
trade_max_percent_of_DQ30_in_last_24_months__with_recent_payment_open_accounts,1.803612e-06,1.632408,18273,4.634982


In [35]:
psi_exp_number = psi_table(experian_train_new, experian_train_old, number_features, 'experian train')

Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.


fit


Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The method of computing PSI metric differs among continuous and categorical variables.
Please be advised that numeric variables that are categorical, such as one-hot-encoded columns (binary variables) should be provided to num_cat_var argument of the class constructor. The met

=== experian train (n=394,241) ===
                                                                                                      psi_overall  psi_changed_rows  n_changed  pct_changed
trade_percent_accounts_with_DQ60_or_greater_in_last_24_months__open_mortgage                               0.0000           22.6108        117       0.0297
trade_percent_accounts_with_DQ120_or_greater_in_last_6_months__non_derog_open_installment                  0.0006           16.9934        399       0.1012
trade_percent_accounts_with_DQ120_or_greater_in_last_6_months__with_recent_payment_open_installment        0.0000           15.6657        124       0.0315
trade_percent_accounts_with_DQ30_or_greater_in_last_12_months__joint_open_installment                      0.0000           15.1760        100       0.0254
trade_percent_accounts_with_DQ30_or_greater_in_last_6_months__non_derog_open_installment                   0.0001           14.2264        664       0.1684
trade_percent_accounts_with_D

In [39]:
psi_exp_number.sort_values(by = ['n_changed', 'psi_overall'], ascending = False).head(50)

,psi_overall,psi_changed_rows,n_changed,pct_changed
trade_number_accounts_with_DQ30_or_greater_in_last_24_months__all_accounts,0.000145,0.443356,4864,1.233763
trade_percent_accounts_with_DQ30_or_greater_in_last_24_months__all_accounts,0.000142,0.796247,4864,1.233763
trade_number_accounts_with_DQ60_or_greater_in_last_24_months__all_accounts,0.000172,0.533301,4391,1.113786
trade_percent_accounts_with_DQ60_or_greater_in_last_24_months__all_accounts,0.000150,0.986627,4391,1.113786
trade_number_accounts_with_DQ30_or_greater_in_last_12_months__all_accounts,0.000166,0.521258,4140,1.050119
trade_percent_accounts_with_DQ30_or_greater_in_last_12_months__all_accounts,0.000160,0.964532,4140,1.050119
trade_number_accounts_with_DQ90_or_greater_in_last_24_months__all_accounts,0.000174,0.576609,4020,1.019681
trade_percent_accounts_with_DQ90_or_greater_in_last_24_months__all_accounts,0.000148,1.109014,4020,1.019681
trade_number_accounts_with_DQ60_or_greater_in_last_12_months__all_accounts,0.000206,0.620123,3815,0.967682
trade_percent_accounts_with_DQ60_or_greater_in_last_12_months__all_accounts,0.000175,1.130845,3815,0.967682


In [41]:
psi_exp_number.sort_values(by ='psi_overall', ascending = False).head(5)

,psi_overall,psi_changed_rows,n_changed,pct_changed
trade_percent_accounts_with_DQ60_or_greater_in_last_24_months__open_education,0.001362,4.844776,2586,0.655944
trade_percent_accounts_with_DQ60_or_greater_in_last_24_months__education,0.000991,3.914255,3134,0.794945
trade_number_accounts_with_DQ120_or_greater_in_last_24_months__individual_open_installment,0.000986,1.495136,1978,0.501724
trade_percent_accounts_with_DQ120_or_greater_in_last_24_months__individual_open_installment,0.000986,4.984062,1978,0.501724
trade_number_accounts_with_DQ120_or_greater_in_last_24_months__active_open_installment,0.000941,1.477796,1984,0.503245
